# **PML Dataframe — Exploratory Data Analysis**

## Exploratory Data Analysis (EDA) Notebook for pml_df Dataframe based on pml_metadata.

### Schema-aligned with:
- Table: `pml.pml_df`
- Metadata: `SELECT * FROM pml.pml_df_metadata pdm WHERE ordinal_position NOTNULL ORDER BY ordinal_position`







In [1]:
%%sql
SELECT * FROM pml.pml_df pd

,ticker,isin,name,description,region,country,trading_country,exchange,unit,sector,...,gross_profit_margin_pct_neg2fqfq,gross_profit_margin_pct_neg3fqfq,gross_profit_margin_pct_neg4fqfq,gross_profit_margin_pct_neg1fy,gross_profit_margin_pct_neg2fy,gross_profit_margin_pct_neg3fy,gross_profit_margin_pct_neg4fy,gross_profit_margin_pct_neg5fy,gross_profit_margin_pct_3yavgfq,gross_profit_margin_pct_5yavgfq
0,NVDA,US67066G1040,NVIDIA Corporation,NVIDIA Corporation operates as a data center s...,United States and Canada,US,US,NasdaqGS,USD,Information Technology,...,0.7341,0.7242,0.6052,0.7499,0.7272,0.5693,0.6493,0.6331,0.7246,0.6797
1,OBCK,DE000BCK2223,Ottobock SE & Co. KGaA,Ottobock SE & Co. KGaA develops produces and d...,Europe,DE,DE,XTRA,EUR,Health Care,...,0.5314,0.5094,0.5156,0.5059,0.4647,0.4473,0.4633,NaN,NaN,NaN
2,CSTM,FR0013467479,Constellium SE,Constellium SE together with its subsidiaries ...,Europe,FR,US,NYSE,USD,Materials,...,0.1450,0.1251,0.1329,0.1310,0.1379,0.1157,0.1079,0.1003,0.1426,0.1246
3,ARM,US0420682058,Arm Holdings plc,Arm Holdings plc researches develops licenses ...,Europe,GB,US,NasdaqGS,USD,Information Technology,...,0.9744,0.9715,0.9774,0.9698,0.9524,0.9604,0.9515,0.9285,0.9648,NaN
4,GOOGL,US02079K3059,Alphabet Inc.,Alphabet Inc. offers various products and plat...,United States and Canada,US,US,NasdaqGS,USD,Communication Services,...,0.5958,0.5951,0.5970,0.5820,0.5663,0.5538,0.5694,0.5358,0.5849,0.5761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6312,TKNSA,TRETKNO00010,Teknosa Iç ve Dis Ticaret Anonim Sirketi,Teknosa Iç ve Dis Ticaret Anonim Sirketi engag...,Africa / Middle East,TR,TR,IBSE,TRY,Consumer Discretionary,...,0.1380,0.1417,0.1255,0.1272,0.1054,0.0857,0.1671,0.1644,0.1209,0.1333
6313,VVEO3,BRVVEOACNOR0,CM Hospitalar S/A,CM Hospitalar S/A engages in the distribution ...,Latin America and Caribbean,BR,BR,BOVESPA,BRL,Health Care,...,0.1467,0.1500,0.1382,0.1127,0.1565,0.1597,0.1765,0.1447,0.1397,0.1504
6314,MICH,EGS38211C016,Misr Chemical Industries Co.,Misr Chemical Industries Co. produces and sell...,Africa / Middle East,EG,EG,CASE,EGP,Materials,...,0.6875,0.6355,0.6901,0.6583,0.6900,0.5559,0.4602,0.3677,0.6680,0.6065
6315,LJQQ3,BRLJQQACNOR5,Lojas Quero-Quero S.A.,Lojas Quero-Quero S.A. engages in the general ...,Latin America and Caribbean,BR,BR,BOVESPA,BRL,Consumer Discretionary,...,0.3243,0.3230,0.3306,0.3481,0.3444,0.3326,0.3904,0.4132,0.3362,0.3460


In [2]:
%%sql
SELECT *
FROM pml.pml_df_metadata pdm
WHERE ordinal_position NOTNULL
ORDER BY ordinal_position

,column_name,category,feature_role,feature_alias,ordinal_position,description,data_type,pymc_role,model_targets,updated_at
0,ticker,identifier,id,ticker,1,Ticker symbol identifier,text,coord,{},2026-06-14 17:30:25.676745
1,isin,identifier,id,isin,2,International Securities Identification Number,text,coord,{},2026-06-14 17:30:25.676745
2,name,identifier,metadata,NaN,3,Company name,text,excluded,{},2026-06-14 17:30:25.676745
3,description,identifier,metadata,NaN,4,Company description / business summary,text,excluded,{},2026-06-14 17:30:25.676745
4,region,classification,categorical,region,5,Geographic region,text,coord,"{kalman_pt,earnings_beat,price_target,credit_r...",2026-06-14 17:30:25.676745
...,...,...,...,...,...,...,...,...,...,...
571,gross_profit_margin_pct_neg3fy,profitability,historical,NaN,572,Lagged gross profit margin % (prior period),double precision,derived_input,{accounting_anomaly},2026-06-14 17:30:25.676745
572,gross_profit_margin_pct_neg4fy,profitability,historical,NaN,573,Lagged gross profit margin % (prior period),double precision,derived_input,{accounting_anomaly},2026-06-14 17:30:25.676745
573,gross_profit_margin_pct_neg5fy,profitability,historical,NaN,574,Lagged gross profit margin % (prior period),double precision,derived_input,{accounting_anomaly},2026-06-14 17:30:25.676745
574,gross_profit_margin_pct_3yavgfq,profitability,predictor,NaN,575,Gross profit margin % (period indicated by suf...,double precision,mutable_predictor,{accounting_anomaly},2026-06-14 17:30:25.676745


## 2. Exploratory Data Analysis (EDA) — `pml_df`

Source: `pml.pml_df` .
We resolve column categories from `pml.pml_df_metadata` and fall back to the
known MV schema where the catalogue has no `kalman_pt` rows yet.

In [3]:
# Build a proper dictionary from the DataFrame: {category: [feature_alias, ...]}
FEATURE_CATEGORIES = pml_metadata.groupby("category")["feature_alias"].apply(list).to_dict()

print(f"Dataset shape: {pml_df.shape[0]} stocks × {pml_df.shape[1]} features")
print(f"Feature categories: {len(FEATURE_CATEGORIES)}")
print(f"\nColumn dtypes:\n{pml_df.dtypes.value_counts()}")

,category,feature_role,n_columns,n_present
0,analyst_ratings,count,6,0
1,analyst_ratings,score,1,0
2,analyst_targets,count,1,0
3,analyst_targets,historical,48,0
4,analyst_targets,target,9,0
5,cash_flow,historical,53,0
6,cash_flow,predictor,23,0
7,classification,categorical,9,0
8,credit_risk,historical,15,0
9,credit_risk,score,3,0
